# SCIKIT-LEARN MASTERY — NOTEBOOK 8
## Chapters 39–45: Semi-Supervised Learning → Composed Estimators → Meta-Estimators → Multi-Class Classification → Multi-Label Classification → Multi-Output Learning → Imbalanced Data

Continuing directly from Notebook 7.

### SCIKIT-LEARN MASTERY PROGRESS
- [x] Chapters 1–38 *(Notebooks 1–7)*
- [x] Semi-Supervised Learning
- [x] Composed Estimators
- [x] Meta-Estimators
- [x] Multi-Class Classification
- [x] Multi-Label Classification
- [x] Multi-Output Learning
- [x] Imbalanced Data


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
np.random.seed(42)


---
# CHAPTER 39 — SEMI-SUPERVISED LEARNING

## 1. Concept
Semi-supervised learning sits between fully supervised (all labels known) and fully unsupervised (no labels): you have a **small amount of labeled data** and a **large amount of unlabeled data**, and want to leverage the unlabeled data's structure to improve on what the small labeled set alone could achieve. This is a common real-world situation — labeling is expensive, but raw data is often abundant.

## 2. The Tools
| Class | Idea |
|---|---|
| `LabelPropagation` | Builds a similarity graph over all points (labeled + unlabeled); labels "propagate" through the graph based on proximity |
| `LabelSpreading` | Similar to LabelPropagation, but more robust to noisy labels (allows some label "regularization" rather than perfectly fixing known labels) |
| `SelfTrainingClassifier` | Wraps *any* supervised classifier: train on labeled data, predict confidently on unlabeled data, add those high-confidence predictions as new "pseudo-labels," retrain, repeat |


In [2]:
from sklearn.semi_supervised import SelfTrainingClassifier, LabelSpreading
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_ss, y_ss_true = make_classification(n_samples=500, n_features=10, n_informative=6, random_state=42)
X_ss_train, X_ss_test, y_ss_train_true, y_ss_test = train_test_split(X_ss, y_ss_true, test_size=0.3, random_state=42)

# Simulate a REALISTIC semi-supervised scenario: only 10% of training labels are actually known
np.random.seed(5)
y_ss_train = y_ss_train_true.copy()
unlabeled_mask = np.random.rand(len(y_ss_train)) > 0.10   # 90% of labels are "unknown"
y_ss_train[unlabeled_mask] = -1   # Scikit-learn's convention: -1 marks an unlabeled sample

print(f"Labeled samples: {(y_ss_train != -1).sum()} / {len(y_ss_train)}")

# BASELINE: train supervised model using ONLY the small labeled subset
baseline = LogisticRegression(max_iter=1000).fit(X_ss_train[y_ss_train != -1], y_ss_train[y_ss_train != -1])
baseline_acc = accuracy_score(y_ss_test, baseline.predict(X_ss_test))

# SELF-TRAINING: leverage the unlabeled data too
self_training = SelfTrainingClassifier(LogisticRegression(max_iter=1000), threshold=0.9)
self_training.fit(X_ss_train, y_ss_train)   # pass ALL data, with -1 marking unlabeled
self_training_acc = accuracy_score(y_ss_test, self_training.predict(X_ss_test))

print(f"\nSupervised baseline (10% labels only):  {baseline_acc:.4f}")
print(f"SelfTrainingClassifier (uses ALL data):  {self_training_acc:.4f}")


Labeled samples: 36 / 350

Supervised baseline (10% labels only):  0.7067
SelfTrainingClassifier (uses ALL data):  0.6867


### Line-by-Line Explanation
- `y_ss_train[unlabeled_mask] = -1` — Scikit-learn's semi-supervised tools use the sentinel value **`-1`** to mark unlabeled samples within the same `y` array — a deliberate, simple convention letting `.fit(X, y)` accept a mix of labeled and unlabeled rows in a single call.
- `SelfTrainingClassifier(LogisticRegression(...), threshold=0.9)` — wraps any base classifier; internally, iterates: fit on currently-labeled data, predict on unlabeled data, promote predictions with confidence `>= threshold` (0.9 here — quite conservative, only very confident pseudo-labels get added) to "labeled" status, and repeat until no more confident predictions remain to add or a max iteration count is reached.
- The baseline uses **only** the 10% genuinely labeled rows; `SelfTrainingClassifier` gets access to the *same* labeled rows plus all the unlabeled ones (used internally for pseudo-labeling) — the accuracy gap (when present) is the direct, quantified value that leveraging unlabeled data added.

### Common Mistakes
- Setting `SelfTrainingClassifier`'s `threshold` too low — accepts low-confidence, likely-wrong pseudo-labels, which then compound: the model trains on its own mistakes, reinforcing them in later iterations (a real risk called "confirmation bias" in the semi-supervised learning literature).
- Assuming semi-supervised learning always beats the pure-supervised baseline — it depends heavily on whether the *unlabeled* data's structure genuinely aligns with the true label boundaries; if the underlying clusters in the unlabeled data don't correspond well to the true classes, pseudo-labeling can actively mislead the model.
- Forgetting the `-1` convention and instead trying to represent "unlabeled" some other way (e.g., `NaN`) — Scikit-learn's semi-supervised tools specifically expect `-1`.

### Best Practices
- Start with a conservative (high) `threshold` for `SelfTrainingClassifier`, and empirically verify (via a genuinely held-out labeled test set, exactly as done above) that self-training actually improves on the supervised-only baseline before trusting it in production.
- Consider semi-supervised methods specifically when labeling is expensive/slow but raw data is abundant — a common situation in medical imaging, some NLP tasks, and specialized industrial applications.

### Interview Insight
**Q: You have 100,000 unlabeled customer records and only 500 labeled ones (labeling required manual review). How might semi-supervised learning help, and what's the risk?**
A: A `SelfTrainingClassifier` (or `LabelPropagation`/`LabelSpreading`) could use the 500 labeled examples as a starting point, then leverage the structure in the 100,000 unlabeled records to improve generalization — effectively getting more value out of the expensive-to-obtain labels by combining them with abundant unlabeled data. The risk is confirmation bias: if early pseudo-labels are wrong (especially likely if `threshold` is too permissive), the model can reinforce its own mistakes over successive iterations, potentially ending up *worse* than the pure 500-example supervised baseline. Always validate empirically against a genuinely held-out labeled test set, exactly as this chapter's baseline comparison does, rather than assuming semi-supervised learning is automatically beneficial.

### Quiz (Chapter 39)
1. What sentinel value marks an unlabeled sample in Scikit-learn's semi-supervised API?
2. How does `SelfTrainingClassifier` work, conceptually, in each iteration?
3. What is "confirmation bias" in this context, and what parameter helps control it?
4. Why must semi-supervised learning's benefit always be verified empirically rather than assumed?


---
# CHAPTER 40 — COMPOSED ESTIMATORS

## 1. Concept
This chapter is a deliberate consolidation checkpoint: every composition tool covered so far — `Pipeline`, `ColumnTransformer`, `FeatureUnion`, `VotingClassifier`, `StackingClassifier`, `BaggingClassifier` — shares one underlying idea: **Scikit-learn estimators can contain other estimators**, and the outer object still honors the exact same `fit`/`predict`/`transform` contract as any single model. This is what lets arbitrarily deep, complex ML systems still be manipulated with the same handful of API calls established in Chapter 2.

## 2. The Full Composition Toolkit, Side by Side


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# A genuinely deep composition: Pipeline( ColumnTransformer( Pipeline(...), Pipeline(...) ), VotingClassifier( Pipeline(...), Pipeline(...) ) )
np.random.seed(40)
n = 600
df_comp = pd.DataFrame({
    "num1": np.random.normal(50, 10, n),
    "num2": np.random.exponential(20, n),
    "cat1": np.random.choice(["A", "B", "C"], n),
})
y_comp = (df_comp["num1"] + df_comp["num2"] > 65).astype(int)

preprocessor_comp = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), ["num1", "num2"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["cat1"]),
])

deep_composition = Pipeline([
    ("preprocessor", preprocessor_comp),
    ("ensemble", VotingClassifier([
        ("lr", LogisticRegression(max_iter=1000)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ], voting="soft")),
])

deep_composition.fit(df_comp, y_comp)
print("Fit succeeded on a 3-level-deep composed estimator.")
print("Test prediction on 3 rows:", deep_composition.predict(df_comp.head(3)))
print("\nDespite the internal depth, we STILL just called .fit(X, y) and .predict(X) --")
print("the exact same two-method contract from Chapter 2, no matter how deep the composition.")


Fit succeeded on a 3-level-deep composed estimator.
Test prediction on 3 rows: [0 0 1]

Despite the internal depth, we STILL just called .fit(X, y) and .predict(X) --
the exact same two-method contract from Chapter 2, no matter how deep the composition.


### Internal Working — Why This Composability Is Possible At All
Every composing object (`Pipeline`, `ColumnTransformer`, `VotingClassifier`, etc.) is itself built to satisfy the Estimator interface from Chapter 2 — it has `.fit()`, exposes `.get_params()`/`.set_params()` (including reaching into nested components via the `__` syntax from Chapter 25), and (where applicable) `.predict()`/`.transform()`. Because a composing object *is itself* a valid estimator, it can be nested inside yet another composing object with zero special-casing — this recursive, self-similar design is Scikit-learn's most important architectural property, and the single biggest payoff of the strict API discipline established all the way back in Chapter 2 and enforced in every custom transformer (Chapter 9).

### Common Mistakes
- Over-composing for its own sake — nesting Pipelines inside FeatureUnions inside Pipelines when a flatter structure would be equally correct and much easier to read/debug; composition depth should be driven by genuine structural need (different columns need different processing, multiple models should vote), not novelty.
- Losing track of parameter names for `GridSearchCV` tuning as composition depth grows — the `__` chain can get long (`ensemble__rf__max_depth`); always verify via `.get_params().keys()` rather than guessing.

### Interview Insight
**Q: What single architectural property of Scikit-learn's design makes deeply nested compositions (Pipelines containing ColumnTransformers containing Pipelines, feeding a VotingClassifier of other Pipelines) possible without any special-casing?**
A: Every composing object (`Pipeline`, `ColumnTransformer`, `FeatureUnion`, `VotingClassifier`, `StackingClassifier`, etc.) is itself implemented to satisfy the exact same Estimator interface (`fit`, `get_params`/`set_params`, and `predict`/`transform` where applicable) as any single model. Because a composing object *is* a valid estimator in its own right, it can always be used anywhere a single estimator could be used — including as one more component inside yet another composing object — with no special-casing required anywhere in the framework. This recursive self-similarity is the direct payoff of the strict, uniform API convention established since Chapter 2.

### Quiz (Chapter 40)
1. What property must every composing object (Pipeline, ColumnTransformer, etc.) itself satisfy for nesting to work?
2. Why is over-composing (nesting for its own sake) a real anti-pattern, not just a style nitpick?


---
# CHAPTER 41 — META-ESTIMATORS

## 1. Concept
A **meta-estimator** specifically *wraps a single estimator* to extend or modify its behavior — distinct from Chapter 40's composition tools (which *combine multiple* estimators). Meta-estimators are how Scikit-learn adds capabilities (multi-class support, calibration, hyperparameter search) to a base model *without modifying the base model's own code at all*.

```
Estimator
   ↓
Meta-estimator (wraps it, adds/modifies behavior)
   ↓
Wrapped estimator (the original, called internally)
```

## 2. The Roster
| Meta-estimator | What it adds |
|---|---|
| `GridSearchCV` / `RandomizedSearchCV` | Hyperparameter search around any estimator (Chapter 25) |
| `OneVsRestClassifier` / `OneVsOneClassifier` | Multi-class support for classifiers that are natively binary-only (Chapter 42) |
| `MultiOutputClassifier` / `MultiOutputRegressor` | Multi-target support for models that natively predict one target at a time (Chapter 44) |
| `CalibratedClassifierCV` | Probability calibration around any classifier (Chapter 31) |
| `BaggingClassifier` | Bootstrap-aggregated ensembling around any base estimator |


In [4]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

X_meta, y_meta = make_classification(n_samples=400, n_features=8, n_classes=3, n_informative=6, random_state=42)

# LinearSVC is NATIVELY binary-focused in its core formulation for some solvers/configs --
# OneVsRestClassifier is the standard meta-estimator wrapper for guaranteed multi-class support
ovr = OneVsRestClassifier(LinearSVC(max_iter=5000))
ovr.fit(X_meta, y_meta)
print("OneVsRestClassifier wraps", ovr.estimator, "\ninternally fits", len(ovr.estimators_), "binary classifiers (one per class)")

# CalibratedClassifierCV wraps ANY classifier to add/correct predict_proba() -- Chapter 31, revisited here
# explicitly through the meta-estimator lens
calibrated_svc = CalibratedClassifierCV(LinearSVC(max_iter=5000), cv=5)
calibrated_svc.fit(X_meta, y_meta)
print("\nCalibratedClassifierCV wraps a model with NO native predict_proba() (LinearSVC)")
print("and ADDS it:", calibrated_svc.predict_proba(X_meta[:2]).round(3))


OneVsRestClassifier wraps LinearSVC(max_iter=5000) 
internally fits 3 binary classifiers (one per class)

CalibratedClassifierCV wraps a model with NO native predict_proba() (LinearSVC)
and ADDS it: [[0.12  0.46  0.42 ]
 [0.319 0.477 0.204]]


### Line-by-Line Explanation
- `OneVsRestClassifier(LinearSVC(...))` — internally fits `n_classes` separate binary classifiers, each trained as "this class vs. all others," then predicts by choosing whichever binary classifier is most confident; `.estimators_` (fitted attribute) exposes the individual binary classifiers directly.
- `CalibratedClassifierCV(LinearSVC(...))` — `LinearSVC` has **no** `predict_proba()` at all natively (unlike `SVC(probability=True)`); wrapping it in `CalibratedClassifierCV` both adds calibrated probability output *and* performs the calibration correctly via internal cross-validation — a meta-estimator can add an entirely new capability, not just modify an existing one.

### The Conceptual Distinction, Restated
- **Composition** (Chapter 40): combine **multiple different** estimators into one (`VotingClassifier([model_a, model_b])`).
- **Meta-estimator** (this chapter): wrap **one** estimator to extend/modify its behavior (`CalibratedClassifierCV(model_a)`).

In practice these categories blur at the edges (a `Pipeline` with one step is arguably both), but the conceptual distinction — "combining many" vs. "wrapping one to add a capability" — is the useful mental model for reasoning about *why* a given tool exists.

### Common Mistakes
- Confusing meta-estimators with composition tools when explaining Scikit-learn's architecture in an interview — genuinely different purposes, even though both rely on the same underlying "estimators can contain estimators" property from Chapter 40.
- Not realizing `GridSearchCV` is itself a meta-estimator wrapping the model being tuned — `grid_search.best_estimator_` is exactly what you'd expect, but `grid_search` *itself* is also a fully valid estimator (has `.fit()`, `.predict()`) that happens to internally search over its wrapped estimator's hyperparameters before settling on one.

### Interview Insight
**Q: Is `GridSearchCV` a composition tool or a meta-estimator, and why does that classification matter conceptually?**
A: It's a meta-estimator — it wraps a *single* base estimator and extends its behavior (adding automatic hyperparameter search around it), rather than combining multiple different estimators into one ensemble. Recognizing this matters because it explains why `GridSearchCV` itself exposes the full Estimator interface (`.fit()`, `.predict()`, `.score()`) — it's not just a utility function that returns a dictionary of results, it *is* a proper estimator in its own right (specifically wrapping and extending the one you gave it), which is exactly why it can itself be used as a step inside a larger Pipeline or composition, per Chapter 40's recursive composability property.

### Quiz (Chapter 41)
1. What's the conceptual difference between a meta-estimator and a composition tool?
2. Why is `CalibratedClassifierCV` a genuinely useful meta-estimator for `LinearSVC` specifically?
3. Why can `GridSearchCV` itself be used as a step inside a Pipeline?


---
# CHAPTER 42 — MULTI-CLASS CLASSIFICATION

## 1. Concept
Most Scikit-learn classifiers (Logistic Regression, Random Forest, Decision Trees, Naive Bayes) handle multi-class problems **natively** — no special wrapping needed. A smaller set of algorithms (classically, SVM-family algorithms) are fundamentally **binary** in their core mathematical formulation and need an explicit multi-class **strategy** layered on top.

## 2. One-vs-Rest vs. One-vs-One
| Strategy | How | Classifiers trained | Typical choice |
|---|---|---|---|
| **One-vs-Rest (OvR)** | Each classifier: "this class" vs. "everything else" | `n_classes` | The default, generally preferred — fewer classifiers, easier probability interpretation |
| **One-vs-One (OvO)** | Each classifier: "this class" vs. "that specific other class," for every pair | `n_classes × (n_classes-1) / 2` | Can be more accurate for some algorithms (notably kernel SVM), at higher computational cost for many classes |


In [5]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import time

X_mc, y_mc = make_classification(n_samples=600, n_features=10, n_classes=5, n_informative=8, random_state=42)
X_mc_train, X_mc_test, y_mc_train, y_mc_test = train_test_split(X_mc, y_mc, test_size=0.3, random_state=42)

print(f"Classes: {sorted(set(y_mc))}  (5 classes)")
print(f"OvR trains: {5} binary classifiers")
print(f"OvO trains: {5*4//2} binary classifiers")
print()

for name, model in [
    ("Native (LogisticRegression, multinomial internally)", LogisticRegression(max_iter=1000)),
    ("Explicit OneVsRestClassifier(SVC)", OneVsRestClassifier(SVC())),
    ("Explicit OneVsOneClassifier(SVC)", OneVsOneClassifier(SVC())),
]:
    start = time.time()
    model.fit(X_mc_train, y_mc_train)
    fit_time = time.time() - start
    acc = model.score(X_mc_test, y_mc_test)
    print(f"{name:<50} acc={acc:.4f}  fit_time={fit_time*1000:.1f}ms")


Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]  (5 classes)
OvR trains: 5 binary classifiers
OvO trains: 10 binary classifiers

Native (LogisticRegression, multinomial internally) acc=0.4944  fit_time=42.8ms
Explicit OneVsRestClassifier(SVC)                  acc=0.7611  fit_time=92.7ms
Explicit OneVsOneClassifier(SVC)                   acc=0.7444  fit_time=75.4ms


### Line-by-Line Explanation
- `LogisticRegression()` — handles multi-class **natively**: modern Scikit-learn automatically uses a genuinely multinomial formulation (not an implicit OvR wrapper) whenever the solver supports it, requiring no explicit wrapping at all.
- `OneVsRestClassifier(SVC())` — explicitly wraps `SVC` (which, note, actually *does* support multi-class internally too via `decision_function_shape`, but this demonstrates the explicit wrapping pattern for algorithms that genuinely need it, like `LinearSVC` in some configurations, or any strictly-binary custom estimator).
- `OneVsOneClassifier(SVC())` — trains `10` pairwise binary classifiers (`5×4/2`) instead of `5`, visibly slower to fit as a direct consequence of training more sub-classifiers.

### Multi-Class Metrics — The `average` Parameter


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

model_mc = LogisticRegression(max_iter=1000).fit(X_mc_train, y_mc_train)
preds_mc = model_mc.predict(X_mc_test)

# Binary metrics (precision, recall, F1) need an "average" strategy to collapse per-class scores into ONE number
for avg in ["macro", "micro", "weighted"]:
    p = precision_score(y_mc_test, preds_mc, average=avg)
    print(f"average='{avg}':  precision={p:.4f}")

print()
print(classification_report(y_mc_test, preds_mc))


average='macro':  precision=0.5011
average='micro':  precision=0.4944
average='weighted':  precision=0.4999

              precision    recall  f1-score   support

           0       0.46      0.55      0.50        33
           1       0.43      0.56      0.49        34
           2       0.57      0.47      0.51        43
           3       0.45      0.36      0.40        42
           4       0.59      0.61      0.60        28

    accuracy                           0.49       180
   macro avg       0.50      0.51      0.50       180
weighted avg       0.50      0.49      0.49       180



### Parameter Explanation — `average` for Multi-Class Metrics
| `average` | Meaning | When to prefer it |
|---|---|---|
| `'macro'` | Unweighted mean of each class's individual score | Every class matters equally, **regardless of size** — good for genuinely imbalanced multi-class problems where the rare class still matters |
| `'micro'` | Aggregate TP/FP/FN across ALL classes first, then compute one score | Effectively weights by class frequency — dominated by the largest class(es) |
| `'weighted'` | Like macro, but weighted by each class's support (sample count) | A middle ground — accounts for imbalance but still reports per-class performance, weighted sensibly |

### Common Mistakes
- Using `average='micro'` on an imbalanced multi-class problem when the minority classes genuinely matter, then being surprised the resulting score is dominated by the majority class's performance — `micro` averaging is mathematically equivalent to accuracy for a single-label multi-class problem, which carries all of accuracy's imbalance-blindness (Chapter 27).
- Explicitly wrapping a classifier that already supports multi-class natively (e.g., `OneVsRestClassifier(LogisticRegression())`) — usually unnecessary, adds complexity, and can even change the mathematical formulation (independent binary classifiers vs. a genuine joint multinomial model) in a way that's sometimes, but not always, worse.
- Not specifying `average=` at all and being surprised by a `ValueError` — several multi-class metrics *require* an explicit `average` choice; there's no sensible default for "how to collapse per-class scores" that Scikit-learn assumes on your behalf.

### Best Practices
- Use `average='macro'` when every class matters equally regardless of frequency (a common choice for genuinely imbalanced multi-class problems).
- Use `average='weighted'` as a reasonable default single-number summary that still accounts for class size sensibly.
- Prefer natively multi-class-capable algorithms/solvers when available; reach for explicit `OneVsRestClassifier`/`OneVsOneClassifier` wrapping specifically for algorithms that genuinely need it.

### Interview Insight
**Q: You report `f1_score(y_true, y_pred, average='micro')` for a 5-class problem with severe class imbalance, and it looks great — but your rarest class is performing terribly. What went wrong with the metric choice?**
A: `average='micro'` aggregates true/false positives/negatives across *all* classes before computing the final score — for single-label multi-class classification, this is mathematically equivalent to overall accuracy, meaning it's dominated by whichever classes have the most samples and is essentially blind to how badly a small minority class performs, exactly like plain accuracy is blind to minority-class performance in binary classification. `average='macro'` (unweighted mean of each class's individual F1) would have revealed the rare class's poor performance directly, since it weights every class equally regardless of its sample count.

### Quiz (Chapter 42)
1. What's the structural difference between One-vs-Rest and One-vs-One, and how many binary classifiers does each train for `k` classes?
2. Why does `average='micro'` behave similarly to accuracy for imbalanced multi-class problems?
3. When would you choose `average='macro'` over `average='weighted'`?


---
# CHAPTER 43 — MULTI-LABEL CLASSIFICATION

## 1. Concept — Multiclass vs Multilabel vs Multitarget, Precisely Distinguished
This terminology is genuinely confusing and a common interview trap — get the distinction exact:

| Term | Meaning | Example |
|---|---|---|
| **Multiclass** | Each sample belongs to **exactly one** of 3+ possible classes | An email is Spam OR Promotions OR Personal (never more than one) |
| **Multilabel** | Each sample can belong to **any number** (0, 1, or many) of possible labels **simultaneously** | A news article tagged both "Politics" AND "Economy" AND "International" at once |
| **Multitarget / Multioutput** | Predicting **multiple separate target variables** at once, each potentially multiclass/regression in its own right | Predicting both a customer's `churn_risk` (classification) AND `expected_lifetime_value` (regression) from the same features |

## 2. Building a Multi-Label Classifier


In [7]:
from sklearn.datasets import make_multilabel_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, jaccard_score

X_ml, Y_ml = make_multilabel_classification(n_samples=500, n_features=10, n_classes=4, n_labels=2, random_state=42)
print("Y_ml shape:", Y_ml.shape, "-- each row can have MULTIPLE 1s (multiple simultaneous labels)")
print("Example rows (each row = which of the 4 labels apply):")
print(Y_ml[:5])

X_ml_train, X_ml_test, Y_ml_train, Y_ml_test = train_test_split(X_ml, Y_ml, test_size=0.3, random_state=42)

# MultiOutputClassifier: trains ONE independent binary classifier PER label
multilabel_model = MultiOutputClassifier(LogisticRegression(max_iter=1000))
multilabel_model.fit(X_ml_train, Y_ml_train)
Y_ml_pred = multilabel_model.predict(X_ml_test)

print("\nhamming_loss (fraction of individual label predictions that are WRONG):", round(hamming_loss(Y_ml_test, Y_ml_pred), 4))
print("jaccard_score (average, samples-based -- overlap between predicted and true label SETS):",
      round(jaccard_score(Y_ml_test, Y_ml_pred, average="samples"), 4))


Y_ml shape: (500, 4) -- each row can have MULTIPLE 1s (multiple simultaneous labels)
Example rows (each row = which of the 4 labels apply):
[[0 1 1 0]
 [1 1 1 1]
 [0 0 0 1]
 [0 1 1 0]
 [1 1 1 0]]

hamming_loss (fraction of individual label predictions that are WRONG): 0.2083
jaccard_score (average, samples-based -- overlap between predicted and true label SETS): 0.6044


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Line-by-Line Explanation
- `MultiOutputClassifier(LogisticRegression())` — trains **one independent binary classifier per label column** internally (label 0's classifier never sees labels 1/2/3 during its own training) — a simple, general-purpose strategy that ignores potential *correlations between labels* (e.g., "Politics" and "International" tags might genuinely co-occur more often than independence would predict) — `ClassifierChain` (a related, more advanced tool) explicitly models these inter-label dependencies by feeding earlier labels' predictions as extra input features for later labels.
- `hamming_loss` — the fraction of **individual label predictions** (not whole samples) that are wrong; a natural multi-label generalization of `1 - accuracy`.
- `jaccard_score(average="samples")` — for each sample, computes the overlap (intersection over union) between the predicted label set and the true label set, then averages across samples — directly measures "how much of the right label SET did we get," rather than treating each label independently.

### Common Mistakes
- Confusing multilabel with multiclass in casual conversation or code — genuinely different problem structures requiring different tools (`OneVsRestClassifier` for multiclass on a binary-only base estimator vs. `MultiOutputClassifier` for multilabel).
- Using `accuracy_score` directly on multi-label predictions without realizing it requires an **exact match** of the entire label set per sample (a very strict, often misleadingly low metric for multilabel) — `hamming_loss` or `jaccard_score` are usually more informative.
- Ignoring label correlations by defaulting to `MultiOutputClassifier`'s fully-independent approach when labels are known to co-occur meaningfully — `ClassifierChain` can capture this at some added complexity/computational cost.

### Interview Insight
**Q: What's the precise difference between multiclass and multilabel classification, and why does it matter for tool selection?**
A: Multiclass means each sample belongs to exactly one of several possible classes (mutually exclusive); multilabel means each sample can simultaneously belong to any number of labels (not mutually exclusive). This distinction directly determines the correct Scikit-learn tools: multiclass uses a single classifier (natively, or wrapped via `OneVsRestClassifier`/`OneVsOneClassifier` if the base algorithm is strictly binary) predicting one label per sample; multilabel requires `MultiOutputClassifier` (or `ClassifierChain`) predicting an independent (or chained) binary decision per possible label, since a sample's true answer isn't a single category but a subset of categories.

### Quiz (Chapter 43)
1. Precisely distinguish multiclass, multilabel, and multitarget/multioutput, with an example of each.
2. Why might `accuracy_score` be a misleadingly strict metric for multilabel classification?
3. What does `ClassifierChain` add over `MultiOutputClassifier`'s independent-per-label approach?


---
# CHAPTER 44 — MULTI-OUTPUT LEARNING

## 1. Concept
`MultiOutputRegressor`/`MultiOutputClassifier` extend a base estimator that natively predicts **one target** into predicting **several targets simultaneously** — each target gets its own independently-fitted copy of the base estimator internally, and predictions for all targets are returned together as one array.

## 2. A Realistic Multi-Target Regression Example


In [8]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Predicting BOTH a customer's next-month spend AND their next-month support-ticket count,
# from the same set of input features -- a genuine multi-target regression scenario
np.random.seed(44)
n_mt = 500
X_mt = np.random.randn(n_mt, 6)
y_spend = X_mt[:, 0] * 50 + X_mt[:, 1] * 20 + np.random.normal(0, 10, n_mt) + 200
y_tickets = np.abs(X_mt[:, 2] * 2 - X_mt[:, 0]) + np.random.normal(0, 0.5, n_mt)
Y_mt = np.column_stack([y_spend, y_tickets])   # shape (n_samples, 2) -- TWO targets

X_mt_train, X_mt_test, Y_mt_train, Y_mt_test = train_test_split(X_mt, Y_mt, test_size=0.3, random_state=42)

# Some models (RandomForestRegressor is one) ALREADY support multi-output natively -- no wrapper needed!
native_multi = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_mt_train, Y_mt_train)
native_preds = native_multi.predict(X_mt_test)
print("RandomForestRegressor handles multi-output NATIVELY -- prediction shape:", native_preds.shape)

# LinearRegression's underlying math ALSO supports multi-output natively (a separate coefficient set per target)
from sklearn.linear_model import LinearRegression
native_linear = LinearRegression().fit(X_mt_train, Y_mt_train)
print("LinearRegression ALSO handles multi-output natively -- coef_ shape:", native_linear.coef_.shape)

for i, name in enumerate(["spend", "tickets"]):
    mae_rf = mean_absolute_error(Y_mt_test[:, i], native_preds[:, i])
    print(f"  RandomForest MAE for '{name}': {mae_rf:.3f}")


RandomForestRegressor handles multi-output NATIVELY -- prediction shape: (150, 2)
LinearRegression ALSO handles multi-output natively -- coef_ shape: (2, 6)
  RandomForest MAE for 'spend': 9.710
  RandomForest MAE for 'tickets': 0.928


### Line-by-Line Explanation
- Note that **both** `RandomForestRegressor` and `LinearRegression` support multi-output prediction **natively** — passing a 2D `Y` (shape `(n_samples, n_targets)`) directly to `.fit()` works without any wrapper at all; `MultiOutputRegressor` is specifically needed only for base estimators that **don't** support this internally (checking a given estimator's documentation for "multi-output" support is the way to know which case you're in).
- `native_linear.coef_.shape` — for multi-output `LinearRegression`, the coefficient array gains an extra dimension: `(n_targets, n_features)` instead of just `(n_features,)`, one full coefficient vector per target, fit **jointly but independently** (no shared learning between targets beyond sharing the same input features).

### When `MultiOutputRegressor` IS Needed


In [9]:
from sklearn.svm import SVR

# SVR (unlike RandomForestRegressor/LinearRegression) does NOT support multi-output natively
try:
    SVR().fit(X_mt_train, Y_mt_train)
except ValueError as e:
    print("SVR alone fails on multi-output Y:")
    print(" ", str(e)[:120])

# MultiOutputRegressor wraps it -- fits ONE independent SVR per target internally
wrapped_svr = MultiOutputRegressor(SVR())
wrapped_svr.fit(X_mt_train, Y_mt_train)
print("\nMultiOutputRegressor(SVR()) succeeds -- internally fits", len(wrapped_svr.estimators_), "independent SVR models")


SVR alone fails on multi-output Y:
  y should be a 1d array, got an array of shape (350, 2) instead.

MultiOutputRegressor(SVR()) succeeds -- internally fits 2 independent SVR models


### Common Mistakes
- Wrapping an already-natively-multi-output-capable estimator (like `RandomForestRegressor`) in `MultiOutputRegressor` unnecessarily — usually harmless (produces the same result, sometimes slightly different due to how each target is fit) but adds needless complexity; check the base estimator's native support first.
- Assuming multi-output learning automatically captures relationships **between** the targets themselves — like `MultiOutputClassifier` (Chapter 43), the wrapper approach fits each target completely independently; if the targets are genuinely correlated (e.g., higher spend often means fewer support tickets) and you want the model to exploit that correlation directly, a genuinely joint multi-output model (some neural network architectures, or specific multi-task learning methods) would be needed instead — this is a real limitation of the simple wrapper approach worth knowing.

### Interview Insight
**Q: Does `MultiOutputRegressor` allow the model to learn relationships BETWEEN the different target variables (e.g., that high spend correlates with low support tickets)?**
A: No — `MultiOutputRegressor` fits a completely independent copy of the base estimator for each target column; each target's model never sees the other targets during training, only the shared input features. Any genuine correlation between targets is not directly exploited by this wrapper approach — it's a convenience for extending single-output estimators to multi-output prediction, not a mechanism for joint/multi-task learning across correlated targets. If exploiting inter-target correlation is specifically desired, a genuinely joint modeling approach (certain neural network architectures, or specialized multi-task learning methods) would be needed instead.

### Quiz (Chapter 44)
1. Name two model types that support multi-output prediction natively, without any wrapper.
2. Why is `MultiOutputRegressor` needed for `SVR` specifically?
3. What's the key limitation of the `MultiOutputRegressor`/`MultiOutputClassifier` wrapping approach regarding relationships between targets?


---
# CHAPTER 45 — IMBALANCED DATA

## 1. Concept
Class imbalance — where one class vastly outnumbers another — has already motivated several earlier chapters (`stratify=` in Chapter 4, PR curves over ROC in Chapter 29, threshold tuning in Chapter 30). This chapter consolidates the **professional handling toolkit** specifically.

## 2. The Toolkit
| Technique | How | Scikit-learn tool |
|---|---|---|
| **Class weighting** | Penalize misclassifying the minority class more heavily during training | `class_weight="balanced"` (most classifiers support this directly) |
| **Sample weighting** | Assign per-sample importance weights explicitly | `sample_weight=` argument to `.fit()` |
| **Threshold tuning** | Move the decision threshold away from the default 0.5 | Manual, using `predict_proba()` (Chapter 30) |
| **Resampling** | Physically change the training set's class balance (oversample minority, undersample majority, or synthesize new minority examples like SMOTE) | **Not built into core Scikit-learn** — the dedicated `imbalanced-learn` library provides this, designed to integrate with Scikit-learn's Pipeline |


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score

X_imb, y_imb = make_classification(n_samples=2000, weights=[0.95, 0.05], random_state=42)   # severe 95/5 imbalance
X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

print(f"Positive class rate: {y_imb_train.mean():.3f}\n")
print(f"{'Approach':<35}{'Precision':>12}{'Recall':>10}{'F1':>10}")

for name, model in [
    ("Default (no weighting)", LogisticRegression(max_iter=1000)),
    ("class_weight='balanced'", LogisticRegression(max_iter=1000, class_weight="balanced")),
]:
    model.fit(X_imb_train, y_imb_train)
    preds = model.predict(X_imb_test)
    print(f"{name:<35}{precision_score(y_imb_test, preds):>12.4f}{recall_score(y_imb_test, preds):>10.4f}{f1_score(y_imb_test, preds):>10.4f}")


Positive class rate: 0.054

Approach                              Precision    Recall        F1
Default (no weighting)                   0.8750    0.4375    0.5833
class_weight='balanced'                  0.2903    0.8438    0.4320


### Interpretation
`class_weight="balanced"` typically **increases recall substantially** (catching more true positives) at some cost to precision (more false positives) — it's directly re-weighting the loss function to penalize missing the minority class more heavily, effectively pushing the model's implicit decision boundary to favor the minority class more than the raw class frequencies alone would produce.

### Internal Working
`class_weight="balanced"` sets each class's weight to `n_samples / (n_classes * class_count)` — inversely proportional to that class's frequency — meaning the rare class's individual misclassifications count proportionally more toward the total loss the optimizer is minimizing, without physically duplicating or removing any actual training rows (unlike resampling).

## 3. `sample_weight` — Even More Granular Control


In [11]:
# sample_weight lets you weight INDIVIDUAL rows, not just whole classes -- e.g., weighting
# recent transactions more heavily than old ones, or weighting high-value customers more heavily
sample_weights = np.where(y_imb_train == 1, 5.0, 1.0)   # manually give the minority class 5x weight (a MANUAL alternative to "balanced")

model_sw = LogisticRegression(max_iter=1000).fit(X_imb_train, y_imb_train, sample_weight=sample_weights)
preds_sw = model_sw.predict(X_imb_test)
print(f"Manual sample_weight (5x on minority): precision={precision_score(y_imb_test, preds_sw):.4f}  recall={recall_score(y_imb_test, preds_sw):.4f}")


Manual sample_weight (5x on minority): precision=0.5250  recall=0.6562


### Why Resampling Must Happen INSIDE the Validation Pipeline
The single most important leakage warning specific to imbalanced data: **any resampling (oversampling/undersampling/SMOTE) must be fit and applied only within each cross-validation fold's training portion**, never on the full dataset before splitting/cross-validating. If a minority-class point is oversampled (duplicated, or has synthetic near-neighbors generated) *before* splitting, near-identical copies of that same point can end up in **both** the training and validation folds — the model then partially "sees" validation-set information during training via these near-duplicates, inflating validation performance in a way that won't hold up on genuinely new data. `imbalanced-learn`'s `Pipeline` (a drop-in extension of Scikit-learn's own `Pipeline` supporting resampling steps) exists specifically to enforce this correctly, exactly mirroring Chapter 11's core leakage-prevention argument.

### Common Mistakes
- Applying `class_weight="balanced"` or manual `sample_weight` and then still evaluating with `accuracy_score` — the weighting changes what the *model* optimizes for, but doesn't change what a chosen *evaluation metric* measures; accuracy remains just as poor a metric choice for imbalanced data regardless of how the model was trained (Chapter 27's lesson still fully applies).
- Resampling before splitting/cross-validating — the specific, serious leakage pattern described above.
- Assuming class weighting and resampling are interchangeable — they address the imbalance from different angles (loss re-weighting vs. physically changing the training distribution) and can even be combined, though that adds complexity best justified by measured benefit.

### Best Practices
- Start with `class_weight="balanced"` — it's free (no extra dependency, built into most classifiers), simple, and often a substantial improvement over doing nothing.
- Reach for `imbalanced-learn`'s resampling techniques (especially SMOTE) when class weighting alone isn't sufficient, always wrapped correctly inside a Pipeline.
- Pair any imbalance-handling technique with the correct evaluation metrics (PR-AUC, recall, F1 — never accuracy alone) from Chapters 27–29.

### Interview Insight
**Q: Why must SMOTE (or any resampling technique) be applied inside each cross-validation fold, rather than once on the whole training set before cross-validation begins?**
A: SMOTE generates synthetic minority-class examples as interpolations between existing minority points' nearest neighbors — if this happens on the full training set before cross-validation splits it into folds, a synthetic point generated from (say) minority example A could easily land in one fold while the original point A itself lands in a different fold used for validation. The validation fold would then contain information derived from a point the model effectively saw a near-duplicate of during training, inflating validation performance in a way that doesn't reflect genuine generalization to truly new data. Applying resampling freshly within each fold's training portion only (as `imbalanced-learn`'s Pipeline enforces) avoids this specific leakage pattern, exactly mirroring the general Pipeline leakage-prevention principle from Chapter 11.

### Quiz (Chapter 45)
1. What does `class_weight="balanced"` actually change about model training, mathematically?
2. Why is `sample_weight` more granular than `class_weight`, and give a use case for that extra granularity.
3. Why must resampling (SMOTE, oversampling) be applied inside each CV fold rather than once beforehand?
4. Does class weighting eliminate the need to choose an appropriate evaluation metric for imbalanced data?

### Chapter 45 Summary
- Class weighting (`class_weight="balanced"`), sample weighting (`sample_weight=`), threshold tuning (Chapter 30), and resampling (via `imbalanced-learn`) are the four core imbalance-handling techniques, addressing the problem from different angles.
- Resampling specifically requires careful Pipeline integration to avoid a serious, specific leakage pattern.
- No imbalance-handling technique replaces the need to choose the right evaluation metric (Chapter 27) — they're complementary, not substitutes.

---

## END OF NOTEBOOK 8 — What You Should Now Be Able To Do
- Apply `SelfTrainingClassifier` to leverage abundant unlabeled data alongside a small labeled set, and verify the benefit empirically rather than assuming it.
- Explain Scikit-learn's recursive composability (why deeply nested Pipelines/ColumnTransformers/VotingClassifiers all still honor `fit`/`predict`) and distinguish composition tools from meta-estimators conceptually.
- Precisely distinguish multiclass, multilabel, and multitarget/multioutput problems and select the correct tool (`OneVsRestClassifier`, `MultiOutputClassifier`, `MultiOutputRegressor`) for each.
- Choose the correct `average=` strategy for multi-class metrics based on whether class imbalance should be masked or exposed.
- Handle imbalanced data professionally via class/sample weighting, and explain precisely why resampling requires careful Pipeline integration to avoid leakage.

## Before Moving to Notebook 9 (Chapters 46–52: Sample Weights, Feature Importance & Interpretability, Permutation Importance, Partial Dependence, Learning Curves, Validation Curves, Model Comparison Framework)